## $\delta^{i,d,v,r,\acute v}$

In [ ]:
def compute_b_idvrv_(env,
                     Rmin,
                     Rmax,
                     nr, 
                     discretization_sigma,
                     gamma,
                     beta,
                     pre
                     ) -> np.ndarray:
    points, width = compute_discretization_grid(Rmin, Rmax, nr)
    new_points = np.linspace(points.min() - points.max(), points.max() - points.min(), nr)
    new_width = new_points[1] - new_points[0]
    sigma = discretization_sigma
    b_idvrv_ = np.ones((env.N, nr, nr, nr, nr)) * -1

    for d in range(nr):
        for v in range(nr):
            for r in range(nr):
                for v_ in range(nr):
                
                    vald = pre*points[r] + gamma*points[v_] - points[v] 
                
                    # gaussian distribution around valq with std `sig` 
                    if d==0:
                        prop = norm.cdf(new_points[d]+new_width/2, vald, sigma)
                    elif d==nr-1: # last intervall
                        prop = 1 - norm.cdf(new_points[d]-new_width/2, vald, sigma)
                    else:
                        prop = norm.cdf(new_points[d]+new_width/2, 
                                        vald, sigma) - norm.cdf(new_points[d]-new_width/2, vald, sigma)
                    
                    b_idvrv_[:,d,v,r,v_] = prop
    return b_idvrv_

NameError: name 'np' is not defined

## Temporal-difference error distribution $D^{i,s,a,d}$

From the reward distribution $R^{i,s,a,r}$ (see Section @sec-ddrlbase), the Q-value distribution $Q^{i,s,a,v}$, and the next-state Q-value distribution ${}^{next}Q^{i,s,a,\acute v}$, we can derive the temporal difference error distribution $D^{i,s,a,d}$.

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Disad(self:DDRLSarsa,
          Xisa  # Behavior profiles
         ) -> jnp.ndarray:
    # Compute needed tensors
    Risar = self.Risar(Xisa)
    Qisaq = self.Qisaq(Xisa)
    nextQisaq = self.NextQisaq(Xisa)
    b_idvrv_ = self.b_idvrv_

    i=0; d=1; v=2; r=3; v_=4; s=5; a=6; s_=7

    Disad = jnp.einsum(b_idvrv_, [i,d,v,r,v_], Risar, [i,s,a,r], nextQisaq, [i,s,a,v_], 
                       Qisaq, [i,s,a,v], [i, s, a, d])
    return Disad

DDRLSarsa.Disad = Disad